### Data merge and feature engineering


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [4]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw = pd.read_csv("../data/isw_processed.csv")
df_isw_matrix = pd.read_csv("../data/matrix_isw.csv")
df_tg = pd.read_csv("../data/telegram_processed.csv")

In [18]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [32]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour
226373,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2023-03-24,14.0,5.4,58.0,0.000,0.0,...,False,False,False,True,False,False,Закарпатська,7,0,0.000000
265162,49.8444,24.0254,Lviv,Europe/Kiev,2023-05-30,18.1,8.3,55.4,0.000,0.0,...,False,False,False,True,False,False,Львівська,13,0,0.000000
453501,48.2924,25.9355,Chernivtsi,Europe/Kiev,2024-04-21,8.6,3.1,69.5,0.753,100.0,...,True,False,False,True,False,False,Чернівецька,24,0,0.000000
228009,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2023-03-26,8.0,6.6,90.7,9.000,100.0,...,True,False,False,False,True,False,Кіровоградська,11,0,0.000000
43376,50.4506,30.5243,Kyiv,Europe/Kiev,2022-05-10,10.9,-3.6,37.7,0.000,0.0,...,False,False,False,True,False,False,Київська,10,0,0.000000
519677,48.6264,22.2851,Uzhgorod,Europe/Uzhgorod,2024-08-14,22.2,11.9,56.4,0.000,0.0,...,False,False,True,False,False,False,Закарпатська,7,0,0.000000
376467,48.0020,37.8145,Donetsk,Europe/Kiev,2023-12-09,-1.2,-1.5,97.7,4.500,100.0,...,False,True,False,True,False,False,Донецька,5,1,37.200000
212294,50.6193,26.2513,Rivne,Europe/Kiev,2023-02-27,-1.9,-5.6,76.4,0.000,0.0,...,False,False,False,True,False,False,Рівненська,17,0,0.000000
554177,50.0042,36.2358,Kharkiv,Europe/Kiev,2024-10-13,7.6,2.0,68.4,1.300,100.0,...,True,False,False,True,False,False,Харківська,20,1,16.450000
5323,49.4168,26.9743,Khmelnytskyi,Europe/Kiev,2022-03-05,-1.1,-4.4,78.6,0.200,100.0,...,False,True,False,True,False,False,Хмельницька,22,0,0.000000


In [26]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,day_conditions_simple_Cloudy,day_conditions_simple_Rain,day_conditions_simple_Snow,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id,alarm_active
0,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,False,True,False,True,False,False,Вінницька,2,0
24,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,False,True,False,True,False,False,Вінницька,2,0
48,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,False,True,False,True,False,False,Вінницька,2,0
72,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,False,True,False,True,False,False,Вінницька,2,0
96,49.2336,28.4486,Vinnytsia,Europe/Kiev,2022-02-24,2.8,-0.3,80.5,0.3,100.0,...,False,False,True,False,True,False,False,Вінницька,2,0


In [33]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 634752 entries, 0 to 634751
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   city_latitude                  634752 non-null  float64       
 1   city_longitude                 634752 non-null  float64       
 2   city_name                      634752 non-null  str           
 3   city_timezone                  634752 non-null  str           
 4   day_datetime                   634752 non-null  datetime64[us]
 5   day_temp                       634752 non-null  float64       
 6   day_dew                        634752 non-null  float64       
 7   day_humidity                   634752 non-null  float64       
 8   day_precip                     634752 non-null  float64       
 9   day_precipprob                 634752 non-null  float64       
 10  day_precipcover                634752 non-null  float64       
 11  day_snow   

In [34]:
df_final['alarms_in_last_24h'] = df_final.groupby('region_id')['alarm_active'].rolling(window=24, min_periods=1).sum().reset_index(level=0, drop=True)
df_final['alarm_in_last_hour'] = df_final.groupby('region_id')['alarm_active'].shift(1)
df_final['total_active_alarms'] = df_final.groupby('datetime_hour')['alarm_active'].transform('sum')
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

df_final.sample(10)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,hour_conditions_simple_Snow,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night
441010,49.8444,24.0254,Lviv,Europe/Kiev,2024-03-30,16.1,6.3,55.0,0.000,0.0,...,False,Львівська,13,0,0.000000,0.0,0.0,10,1,0
324007,48.9226,24.7147,Ivano-Frankivsk,Europe/Kiev,2023-09-09,17.9,12.2,73.0,0.000,0.0,...,False,Івано-Франківська,9,0,0.000000,0.0,0.0,3,1,0
614579,46.9734,31.9852,Mykolaiv,Europe/Kiev,2025-01-25,6.5,6.1,97.4,1.100,100.0,...,False,Миколаївська,14,1,60.000000,5.0,1.0,11,1,1
205418,48.4753,35.0160,Dnipro,Europe/Kiev,2023-02-15,0.1,-3.8,75.1,1.000,100.0,...,True,Дніпропетровська,4,0,0.000000,1.0,0.0,5,0,0
623595,48.0020,37.8145,Donetsk,Europe/Kiev,2025-02-10,-7.8,-11.2,77.4,0.000,0.0,...,False,Донецька,5,1,19.216667,20.0,1.0,3,0,0
169107,48.0020,37.8145,Donetsk,Europe/Kiev,2022-12-14,-1.0,-4.7,76.3,0.000,0.0,...,False,Донецька,5,1,33.083333,4.0,0.0,4,0,0
267765,48.2924,25.9355,Chernivtsi,Europe/Kiev,2023-06-03,14.9,6.4,60.4,0.003,100.0,...,False,Чернівецька,24,0,0.000000,0.0,0.0,4,1,0
230917,49.5879,34.5517,Poltava,Europe/Kiev,2023-03-31,4.3,-3.4,60.1,0.000,0.0,...,False,Полтавська,16,0,0.000000,4.0,0.0,7,0,0
152419,49.4168,26.9743,Khmelnytskyi,Europe/Kiev,2022-11-15,4.7,3.2,90.1,0.000,0.0,...,False,Хмельницька,22,1,15.800000,1.0,0.0,24,0,0
151539,48.0020,37.8145,Donetsk,Europe/Kiev,2022-11-14,4.9,0.5,74.0,0.000,0.0,...,False,Донецька,5,1,42.866667,14.0,0.0,3,0,1


In [39]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(index='datetime_hour', columns='region_id', values='alarm_active', fill_value=0)

def get_neighbour_sum_safe(row):
    region_id = row['region_id']
    datetime = row['datetime_hour']
    
    if region_id in neighbouring_regions:
        neighbours = neighbouring_regions[region_id]
        
        if datetime in alarms_matrix.index:
            valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]

            if valid_neighbours:
                return alarms_matrix.loc[datetime, valid_neighbours].sum()
    return 0

df_final['neighbour_alarms'] = df_final.apply(get_neighbour_sum_safe, axis=1)

df_final.sample(15)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,region_key,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,neighbour_alarms
135711,50.9080,34.7972,Sumy,Europe/Kiev,2022-10-17,6.0,1.0,72.9,0.0,0.0,...,Сумська,18,1,10.366667,9.0,1.0,24,0,0,3.0
467193,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2024-05-15,11.3,1.0,50.9,0.0,0.0,...,Кіровоградська,11,0,0.000000,3.0,0.0,1,0,1,0.0
122120,50.4506,30.5243,Kyiv,Europe/Kiev,2022-09-24,9.9,6.8,81.7,0.0,0.0,...,Київська,10,0,0.000000,0.0,0.0,10,1,1,2.0
462801,48.5085,32.2656,Kropyvnytskyi,Europe/Kiev,2024-05-07,16.3,9.5,66.4,10.1,100.0,...,Кіровоградська,11,0,0.000000,2.0,0.0,2,0,0,0.0
164347,49.4168,26.9743,Khmelnytskyi,Europe/Kiev,2022-12-06,-3.6,-4.9,90.9,0.4,100.0,...,Хмельницька,22,0,0.000000,4.0,0.0,0,0,0,0.0
383250,46.6401,32.6142,Kherson,Europe/Kiev,2023-12-21,6.3,5.4,93.9,0.0,0.0,...,Херсонська,21,0,0.000000,15.0,0.0,3,0,0,1.0
145119,50.9080,34.7972,Sumy,Europe/Kiev,2022-11-02,5.3,0.1,69.9,1.0,100.0,...,Сумська,18,0,0.000000,4.0,0.0,6,0,0,2.0
502499,46.9734,31.9852,Mykolaiv,Europe/Kiev,2024-07-15,31.4,17.0,43.6,0.0,0.0,...,Миколаївська,14,0,0.000000,5.0,0.0,4,0,0,1.0
439034,48.4753,35.0160,Dnipro,Europe/Kiev,2024-03-27,7.4,4.1,81.5,0.0,0.0,...,Дніпропетровська,4,0,0.000000,6.0,0.0,0,0,1,0.0
403817,50.0042,36.2358,Kharkiv,Europe/Kiev,2024-01-26,0.5,0.4,99.2,1.4,100.0,...,Харківська,20,1,18.100000,8.0,1.0,11,0,1,4.0


In [54]:
def calculate_hours_since_last(series):
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last)

In [52]:
df_final['neighbour_alarms'] = df_final['neighbour_alarms'].astype(int)
df_final['alarms_in_last_24h'] = df_final['alarms_in_last_24h'].astype(int)
df_final['alarm_in_last_hour'] = df_final['alarm_in_last_hour'].fillna(0).astype(int)

In [55]:
df_final.sample(20)

,city_latitude,city_longitude,city_name,city_timezone,day_datetime,day_temp,day_dew,day_humidity,day_precip,day_precipprob,...,region_id,alarm_active,alarm_minutes_in_hour,alarms_in_last_24h,alarm_in_last_hour,total_active_alarms,is_weekend,is_night,neighbour_alarms,hours_since_last_alarm
76178,48.4753,35.0160,Dnipro,Europe/Kiev,2022-07-06,28.3,14.1,45.1,0.806,100.0,...,4,0,0.000000,4,0,2,0,0,2,4
424361,50.0042,36.2358,Kharkiv,Europe/Kiev,2024-03-01,1.0,-2.8,76.8,0.000,0.0,...,20,0,0.000000,11,0,2,0,0,2,3
441423,50.9080,34.7972,Sumy,Europe/Kiev,2024-03-31,16.0,6.1,53.4,0.000,0.0,...,18,0,0.000000,13,0,9,1,0,1,2
468903,50.9080,34.7972,Sumy,Europe/Kiev,2024-05-18,15.5,9.5,71.4,1.026,100.0,...,18,1,60.000000,7,1,7,1,1,2,0
624682,49.8444,24.0254,Lviv,Europe/Kiev,2025-02-12,-5.4,-8.9,77.8,0.000,0.0,...,13,0,0.000000,0,0,3,0,0,0,30
521009,50.0042,36.2358,Kharkiv,Europe/Kiev,2024-08-16,20.2,8.3,49.7,0.000,0.0,...,20,1,60.000000,21,1,3,0,0,2,0
386851,49.4168,26.9743,Khmelnytskyi,Europe/Kiev,2023-12-27,4.3,0.2,75.0,1.000,100.0,...,22,0,0.000000,7,0,4,0,0,0,2
314378,48.4753,35.0160,Dnipro,Europe/Kiev,2023-08-23,20.3,12.6,63.4,0.000,0.0,...,4,0,0.000000,8,1,0,0,0,0,1
478620,46.4725,30.7371,Odesa,Europe/Kiev,2024-06-03,23.9,14.8,59.6,0.000,0.0,...,15,0,0.000000,4,0,6,0,1,0,7
446728,49.5527,25.5889,Ternopil,Europe/Kiev,2024-04-09,17.1,7.7,57.0,0.000,0.0,...,19,0,0.000000,4,0,3,0,0,0,9


In [63]:
df_isw_matrix.sample(10)

,date,isw_tfidf_abandon,isw_tfidf_abandoned,isw_tfidf_ability,isw_tfidf_able,isw_tfidf_abroad,isw_tfidf_accept,isw_tfidf_access,isw_tfidf_access archive,isw_tfidf_accession,...,isw_tfidf_zelene pole,isw_tfidf_zelensky,isw_tfidf_zelensky stated,isw_tfidf_zelenyi,isw_tfidf_zelenyi hai,isw_tfidf_zhytomyr,isw_tfidf_znpp,isw_tfidf_zone,isw_tfidf_zone northern,isw_tfidf_zvirove
824,2024-06-02,0.009579,0.000000,0.021131,0.017742,0.000000,0.000000,0.003808,0.004194,0.0,...,0.000000,0.038362,0.000000,0.000000,0.000000,0.000000,0.000000,0.004591,0.00000,0.000000
1061,2025-01-29,0.036434,0.000000,0.008037,0.004218,0.000000,0.014903,0.003621,0.003988,0.0,...,0.000000,0.018240,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.014372
904,2024-08-21,0.000000,0.000000,0.007402,0.003885,0.007221,0.006863,0.003335,0.003673,0.0,...,0.007336,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
1407,2026-01-13,0.000000,0.000000,0.016470,0.003457,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.014950,0.005280,0.000000,0.000000,0.006763,0.000000,0.017891,0.00000,0.000000
1452,2026-02-27,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.013005,0.00000,0.000000
1344,2025-11-08,0.000000,0.000000,0.004772,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.016245,0.000000,0.008610,0.008821,0.000000,0.000000,0.010368,0.00806,0.008534
1069,2025-02-06,0.000000,0.000000,0.008382,0.000000,0.000000,0.000000,0.003776,0.004159,0.0,...,0.000000,0.014265,0.000000,0.015121,0.015493,0.008604,0.000000,0.000000,0.00000,0.007494
592,2023-10-11,0.000000,0.000000,0.018931,0.004967,0.009234,0.017551,0.008529,0.004697,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.066542,0.000000,0.00000,0.000000
448,2023-05-20,0.000000,0.000000,0.006169,0.006474,0.000000,0.000000,0.005559,0.006122,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
601,2023-10-20,0.000000,0.000000,0.000000,0.005744,0.000000,0.010148,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000


In [70]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

KeyError: 'date'

In [71]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime'])
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime'])

In [ ]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

MemoryError: Unable to allocate 4.85 MiB for an array with shape (635328, 1) and data type float64